# Player Bio Download

Raw enrichment notebook for player biographical attributes. This workflow is intentionally independent from ranking logic so it can be rerun/versioned as a standalone data layer.


In [ ]:
from __future__ import annotations
from datetime import date
from pathlib import Path
import json
import urllib.error
import urllib.parse
import urllib.request
import pandas as pd
from IPython.display import display
from pybaseball import batting_stats, pitching_stats, cache, chadwick_register
cache.enable()


## Configuration


In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATE_TAG = date.today().strftime('%Y%m%d')
CURRENT_YEAR = date.today().year
YEARS_TO_LOAD = [CURRENT_YEAR - 2, CURRENT_YEAR - 1, CURRENT_YEAR]
INCLUDE_STATUS_POOL = True

STATS_API_BASE = 'https://statsapi.mlb.com/api/v1'
PEOPLE_BATCH_SIZE = 100

YEARS_TO_LOAD


## 1) Load player universe (leaderboards + optional active status draft pool)


In [ ]:
def fetch_leaderboard_ids(year: int) -> pd.DataFrame:
    hitters = batting_stats(year, qual=0)
    pitchers = pitching_stats(year, qual=0)

    id_frames = []
    for source_name, frame in [('hitter_lb', hitters), ('pitcher_lb', pitchers)]:
        if frame.empty or 'IDfg' not in frame.columns:
            continue
        tmp = frame[['IDfg', 'Name']].copy()
        tmp['source'] = source_name
        tmp['season'] = year
        id_frames.append(tmp)

    if not id_frames:
        return pd.DataFrame(columns=['fangraphs_id', 'full_name_lb', 'source', 'season'])

    out = pd.concat(id_frames, ignore_index=True)
    out = out.rename(columns={'IDfg': 'fangraphs_id', 'Name': 'full_name_lb'})
    out['fangraphs_id'] = pd.to_numeric(out['fangraphs_id'], errors='coerce').astype('Int64')
    return out.dropna(subset=['fangraphs_id'])


In [ ]:
leaderboard_ids = pd.concat([fetch_leaderboard_ids(year) for year in YEARS_TO_LOAD], ignore_index=True)
leaderboard_ids = leaderboard_ids.drop_duplicates(['fangraphs_id'])

print(f'Unique Fangraphs IDs from leaderboards: {len(leaderboard_ids):,}')
leaderboard_ids.head()


In [ ]:
def load_latest_status_pool_ids(data_dir: Path) -> pd.DataFrame:
    candidates = sorted(data_dir.glob('player_status_roster_snapshot_*.parquet'))
    if not candidates:
        return pd.DataFrame(columns=['mlbam_id', 'status_snapshot_source'])

    latest = candidates[-1]
    frame = pd.read_parquet(latest)
    if 'person.id' not in frame.columns:
        return pd.DataFrame(columns=['mlbam_id', 'status_snapshot_source'])

    out = frame[['person.id']].dropna().drop_duplicates().copy()
    out = out.rename(columns={'person.id': 'mlbam_id'})
    out['mlbam_id'] = pd.to_numeric(out['mlbam_id'], errors='coerce').astype('Int64')
    out['status_snapshot_source'] = latest.name
    return out.dropna(subset=['mlbam_id'])

status_pool_ids = load_latest_status_pool_ids(DATA_DIR) if INCLUDE_STATUS_POOL else pd.DataFrame(columns=['mlbam_id'])
print(f'Unique MLBAM IDs from status snapshot: {len(status_pool_ids):,}')
status_pool_ids.head()


## 2) Pull bio attributes


In [ ]:
register = chadwick_register()
register = register[['key_mlbam', 'key_fangraphs', 'name_first', 'name_last']].copy()
register['mlbam_id'] = pd.to_numeric(register['key_mlbam'], errors='coerce').astype('Int64')
register['fangraphs_id'] = pd.to_numeric(register['key_fangraphs'], errors='coerce').astype('Int64')
register['full_name_register'] = (register['name_first'].fillna('') + ' ' + register['name_last'].fillna('')).str.strip()
register = register[['mlbam_id', 'fangraphs_id', 'full_name_register']].dropna(subset=['mlbam_id'])

mlbam_from_lb = register.merge(leaderboard_ids[['fangraphs_id']], how='inner', on='fangraphs_id')
mlbam_from_lb = mlbam_from_lb[['mlbam_id', 'fangraphs_id', 'full_name_register']]

universe_ids = pd.concat([
    mlbam_from_lb[['mlbam_id']],
    status_pool_ids[['mlbam_id']] if not status_pool_ids.empty else pd.DataFrame(columns=['mlbam_id']),
], ignore_index=True).dropna().drop_duplicates()

universe_ids['mlbam_id'] = pd.to_numeric(universe_ids['mlbam_id'], errors='coerce').astype('Int64')
universe_ids = universe_ids.dropna(subset=['mlbam_id']).sort_values('mlbam_id').reset_index(drop=True)

print(f'Universe players (unique MLBAM IDs): {len(universe_ids):,}')


In [ ]:
def fetch_people_batch(mlbam_ids: list[int]) -> pd.DataFrame:
    if not mlbam_ids:
        return pd.DataFrame()

    params = urllib.parse.urlencode({'personIds': ','.join(str(i) for i in mlbam_ids), 'hydrate': 'currentTeam'})
    url = f'{STATS_API_BASE}/people?{params}'
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.loads(response.read().decode('utf-8'))

    rows = []
    for p in payload.get('people', []):
        rows.append({
            'mlbam_id': p.get('id'),
            'full_name': p.get('fullName'),
            'birth_date': p.get('birthDate'),
            'bats': (p.get('batSide') or {}).get('code'),
            'throws': (p.get('pitchHand') or {}).get('code'),
            'birth_country': p.get('birthCountry'),
            'birth_city': p.get('birthCity'),
            'debut_date': p.get('mlbDebutDate'),
        })
    return pd.DataFrame(rows)

people_batches = []
ids_list = [int(v) for v in universe_ids['mlbam_id'].dropna().astype(int).tolist()]
for start in range(0, len(ids_list), PEOPLE_BATCH_SIZE):
    chunk = ids_list[start:start + PEOPLE_BATCH_SIZE]
    try:
        people_batches.append(fetch_people_batch(chunk))
    except urllib.error.HTTPError as exc:
        print(f'HTTPError for chunk starting at {start}: {exc.code}')
    except urllib.error.URLError as exc:
        print(f'URLError for chunk starting at {start}: {exc.reason}')

bio_raw = pd.concat(people_batches, ignore_index=True) if people_batches else pd.DataFrame()
print(f'Pulled bio rows: {len(bio_raw):,}')
bio_raw.head()


## 3) Normalize to a stable schema


In [ ]:
player_bio = bio_raw.copy()
player_bio['mlbam_id'] = pd.to_numeric(player_bio['mlbam_id'], errors='coerce').astype('Int64')

player_bio = player_bio.merge(
    register[['mlbam_id', 'fangraphs_id']].drop_duplicates('mlbam_id'),
    on='mlbam_id',
    how='left'
)

player_bio['birth_date'] = pd.to_datetime(player_bio['birth_date'], errors='coerce')
player_bio['debut_date'] = pd.to_datetime(player_bio['debut_date'], errors='coerce')

today = pd.Timestamp(date.today())
player_bio['age'] = ((today - player_bio['birth_date']).dt.days / 365.25).round(2)

stable_cols = [
    'mlbam_id', 'fangraphs_id', 'full_name',
    'birth_date', 'age',
    'bats', 'throws',
    'birth_country', 'birth_city',
    'debut_date',
]

player_bio = (
    player_bio[stable_cols]
    .drop_duplicates(subset=['mlbam_id'])
    .sort_values(['mlbam_id'])
    .reset_index(drop=True)
)

player_bio.head()


## 4) Basic quality checks


In [ ]:
missing_rates = (
    player_bio.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_rate')
    .to_frame()
)

valid_handedness = {'L', 'R', 'S'}
invalid_bats = sorted(set(player_bio['bats'].dropna()) - valid_handedness)
invalid_throws = sorted(set(player_bio['throws'].dropna()) - {'L', 'R'})

print('Missing rates:')
display(missing_rates)
print('Invalid bats values:', invalid_bats)
print('Invalid throws values:', invalid_throws)


## 5) Save artifact


In [ ]:
out_path = DATA_DIR / f'player_bio_{DATE_TAG}.parquet'
player_bio.to_parquet(out_path, index=False)
print(f'Wrote {out_path} ({len(player_bio):,} rows)')
